In [20]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import json

In [21]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)
sd2d['บ้านใหม่'] = 'วังเหนือ'
sd2d['แจ้ห่ม(ในเขต)'] = "แจ้ห่ม"
sd2d['แจ้ห่ม(นอกเขต)'] = "แจ้ห่ม"

all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

# Data Extraction

In [22]:
url = "https://www.thaiphc.net/phc/phcadmin/administrator/Report/OSMRP000VHV.php"
init_body = {
    "provinceid": 52,
    "ampurid": "",
    "tambonid": "",
    "search": "ออกรายงาน" 
}

In [23]:
response = requests.post(url=url, data=init_body)
response.encoding = 'tis-620'

In [24]:
if response.status_code == 200:
    html = response.text

In [25]:
district2id = dict()
sub_district2val = dict()

In [26]:
soup = BeautifulSoup(html, "lxml")
for district in all_distrcit:
    option = soup.select_one(f"option:-soup-contains('{district}')")
    if option:
        val = option.get("value")
        district2id[district] = val
    else:
        district2id[district] = -1

In [27]:
all_subdistrict.append("แจ้ห่ม")

In [28]:
for district, id in district2id.items():
    if id == -1:continue
    print(district)
    init_body['ampurid'] = int(id)
    response = requests.post(url=url, data=init_body)
    response.encoding = 'tis-620'
    if response.status_code == 200:
        html = response.text
        soup = BeautifulSoup(html, "lxml")
        tables = soup.select("tr>td.tableb")
        for i, row in enumerate(tables):
            a = row.select_one("a")
            if a and a.text.strip() in all_subdistrict:
                record = tables[i+1:i+6]
                sub_district2val[a.text.strip()] = dict()
                sub_district2val[a.text.strip()]['สมัครเป็นจิตอาสาแล้ว'] = int(record[0].text.strip())
                sub_district2val[a.text.strip()]['จะสมัครเป็นจิตอาส'] = int(record[1].text.strip())
                sub_district2val[a.text.strip()]['ยังไม่พร้อมเป็นจิตอาสา'] = int(record[2].text.strip())
                sub_district2val[a.text.strip()]['ยังไม่ได้เลือกสถานะจิตอาสา'] = int(record[3].text.strip())
                sub_district2val[a.text.strip()]['จำนวนจิตอาสารวม'] = int(record[4].text.strip())
    else:
        print(f"Something went wrong with code: {response.status_code}")

เมืองลำปาง
งาว
แจ้ห่ม
วังเหนือ
เมืองปาน


In [29]:
for sd in all_subdistrict:
    if sd not in sub_district2val and "ชุดที่" not in sd and "แจ้ห่ม" not in sd:
        sub_district2val[sd] = dict()
        sub_district2val[sd]['สมัครเป็นจิตอาสาแล้ว'] = -1
        sub_district2val[sd]['จะสมัครเป็นจิตอาส'] = -1
        sub_district2val[sd]['ยังไม่พร้อมเป็นจิตอาสา'] = -1
        sub_district2val[sd]['ยังไม่ได้เลือกสถานะจิตอาสา'] = -1
        sub_district2val[sd]['ยังไม่ได้เลือกสถานะจิตอาสา'] = -1

In [30]:
sub_district2val

{'บ้านแลง': {'สมัครเป็นจิตอาสาแล้ว': 179,
  'จะสมัครเป็นจิตอาส': 0,
  'ยังไม่พร้อมเป็นจิตอาสา': 1,
  'ยังไม่ได้เลือกสถานะจิตอาสา': 7,
  'จำนวนจิตอาสารวม': 187},
 'บ้านเสด็จ': {'สมัครเป็นจิตอาสาแล้ว': 275,
  'จะสมัครเป็นจิตอาส': 10,
  'ยังไม่พร้อมเป็นจิตอาสา': 0,
  'ยังไม่ได้เลือกสถานะจิตอาสา': 12,
  'จำนวนจิตอาสารวม': 299},
 'นาแก': {'สมัครเป็นจิตอาสาแล้ว': 72,
  'จะสมัครเป็นจิตอาส': 3,
  'ยังไม่พร้อมเป็นจิตอาสา': 0,
  'ยังไม่ได้เลือกสถานะจิตอาสา': 30,
  'จำนวนจิตอาสารวม': 105},
 'บ้านโป่ง': {'สมัครเป็นจิตอาสาแล้ว': 170,
  'จะสมัครเป็นจิตอาส': 1,
  'ยังไม่พร้อมเป็นจิตอาสา': 0,
  'ยังไม่ได้เลือกสถานะจิตอาสา': 10,
  'จำนวนจิตอาสารวม': 181},
 'บ้านร้อง': {'สมัครเป็นจิตอาสาแล้ว': 183,
  'จะสมัครเป็นจิตอาส': 14,
  'ยังไม่พร้อมเป็นจิตอาสา': 2,
  'ยังไม่ได้เลือกสถานะจิตอาสา': 1,
  'จำนวนจิตอาสารวม': 200},
 'บ้านหวด': {'สมัครเป็นจิตอาสาแล้ว': 86,
  'จะสมัครเป็นจิตอาส': 23,
  'ยังไม่พร้อมเป็นจิตอาสา': 0,
  'ยังไม่ได้เลือกสถานะจิตอาสา': 2,
  'จำนวนจิตอาสารวม': 111},
 'บ้านแหง': {'สมัครเป็นจิตอาส

In [35]:
df_osm = pd.DataFrame(sub_district2val).T
df_osm.to_csv("../master_osmpertumbon.csv")

# Data Preparation

In [38]:
import pandas as pd
df_results = pd.read_csv("../master_result_clean.csv")
df_results.head(10)

,type,province,district,sub-district,unit_number,name,score,score_clean
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ไทยทรัพย์ทวี,0,0
1,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,เพื่อชาติไทย,6,6
2,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ใหม่,4,4
3,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,มิติใหม่,0,0
4,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,รวมใจไทย,1,1
5,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,รวมไทยสร้างชาติ,9,9
6,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,พลวัต,1,1
7,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ประชาธิปไตยใหม่,1,1
8,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,เพื่อไทย,6,6
9,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ทางเลือกใหม่,3,3


In [46]:
results_groupby = df_results.groupby(['district', 'name'])['score_clean'].sum().unstack("name")

In [49]:
cols = results_groupby.columns.tolist()

In [65]:
df_1 = pd.read_csv("../master_summary.csv", index_col=0)
mannual_index = df_1.loc[df_1['valid_ballots'] == -1, :].index.tolist()
mannual_sd = df_1.loc[df_1['valid_ballots'] == -1, ['type', 'unit_number', 'district', 'sub-district']].values.tolist()

In [67]:
for l in mannual_sd:
    valid_ball = df_results[(df_results['type'] == l[0]) & (df_results['unit_number'] == l[1]) & (df_results['district'] == l[2]) & (df_results['sub-district'] == l[3])]['score_clean'].sum()
    df_1.loc[(df_1['type'] == l[0]) & (df_1['unit_number'] == l[1]) & (df_1['district'] == l[2]) & (df_1['sub-district'] == l[3]), 'valid_ballots'] = valid_ball

In [68]:
df_1.loc[df_1['valid_ballots'] == -1, :]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots


In [69]:
df_1.to_csv("../master_summary1.csv", index=False)